# Ch01-01: 다변량 데이터 시각화와 차원 해석 — 모범답안


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
np.random.seed(42)

---
## 문제 1 풀이

PCA 기반 변수 순서 재배열로 앤드류스 곡선 분리 개선 확인

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pandas.plotting import parallel_coordinates, andrews_curves
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
means = [np.array([2,0,-1,3,1,-2]), np.array([-1,3,2,-1,0,4]), np.array([0,-2,1,0,3,-1])]
cov = np.eye(6)*0.8
for i in range(6):
    for j in range(6):
        if i!=j: cov[i,j] = 0.3*((-1)**(i+j))
data = np.vstack([np.random.multivariate_normal(m, cov, 100) for m in means])
labels = np.repeat(['A','B','C'], 100)
cols = [f'X{i}' for i in range(1,7)]
df = pd.DataFrame(data, columns=cols); df['cluster'] = labels

fig, ax = plt.subplots(figsize=(12,5))
parallel_coordinates(df, 'cluster', ax=ax, alpha=0.3)
ax.set_title('평행좌표: 6차원 3클러스터'); plt.tight_layout(); plt.show()

pca = PCA(); pca.fit(StandardScaler().fit_transform(data))
order = np.argsort(np.abs(pca.components_[0]))[::-1]
new_cols = [cols[i] for i in order]

fig, axes = plt.subplots(1,2,figsize=(16,5))
andrews_curves(df[cols+['cluster']], 'cluster', ax=axes[0], alpha=0.2)
axes[0].set_title('원래 순서')
andrews_curves(df[new_cols+['cluster']], 'cluster', ax=axes[1], alpha=0.2)
axes[1].set_title(f'PCA 순서: {new_cols}')
plt.tight_layout(); plt.show()


**결과 해석**: PCA 주성분 기여도 순으로 변수를 재배열하면 분산이 큰 성분이 저주파 항에 배치되어 클러스터 분리가 시각적으로 향상된다.

---
## 문제 2 풀이

Classical MDS 직접 구현 + Swiss Roll에서 유클리드 vs Isomap 비교

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.datasets import make_swiss_roll
from scipy.spatial.distance import pdist, squareform
from scipy.sparse.csgraph import shortest_path
from sklearn.neighbors import NearestNeighbors

X_sr, color = make_swiss_roll(1000, noise=0.5, random_state=42)

def classical_mds(D, n_components=2):
    n = D.shape[0]
    H = np.eye(n) - np.ones((n,n))/n
    B = -0.5 * H @ (D**2) @ H; B = (B+B.T)/2
    eigvals, eigvecs = np.linalg.eigh(B)
    idx = np.argsort(eigvals)[::-1]
    eigvals, eigvecs = eigvals[idx], eigvecs[:,idx]
    pos = np.maximum(eigvals[:n_components], 0)
    return eigvecs[:,:n_components]*np.sqrt(pos), eigvals

D_euc = squareform(pdist(X_sr, 'euclidean'))
Y_euc, ev_euc = classical_mds(D_euc, 2)

nn = NearestNeighbors(n_neighbors=10).fit(X_sr)
G = nn.kneighbors_graph(mode='distance')
D_geo = shortest_path(G, method='D', directed=False)
Y_iso, ev_iso = classical_mds(D_geo, 2)

fig, axes = plt.subplots(1,3,figsize=(18,5))
axes[0].scatter(X_sr[:,0], X_sr[:,2], c=color, cmap='Spectral', s=5)
axes[0].set_title('Swiss Roll (원본)')
axes[1].scatter(Y_euc[:,0], Y_euc[:,1], c=color, cmap='Spectral', s=5)
axes[1].set_title('유클리드 MDS')
axes[2].scatter(Y_iso[:,0], Y_iso[:,1], c=color, cmap='Spectral', s=5)
axes[2].set_title('Isomap (측지선 MDS)')
plt.tight_layout(); plt.show()

def stress(D_orig, Y):
    d = squareform(pdist(Y))
    t = np.triu_indices_from(D_orig, k=1)
    return np.sqrt(np.sum((D_orig[t]-d[t])**2)/np.sum(D_orig[t]**2))
print(f"유클리드 MDS Stress: {stress(D_euc, Y_euc):.4f}")
print(f"Isomap Stress: {stress(D_geo, Y_iso):.4f}")


**결과 해석**: Swiss Roll은 2차원 다양체이므로 유클리드 MDS는 구조를 보존 못 한다. 측지선 거리 기반 Isomap은 펼쳐진 구조를 복원한다.

---
## 문제 3 풀이

Coplot 구현 및 Simpson's paradox 탐색

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
np.random.seed(42)

def coplot(df, x_col, y_col, z_col, n_panels=6, overlap=0.3):
    z = df[z_col].values; z_min, z_max = z.min(), z.max()
    step = (z_max - z_min) / (n_panels - (n_panels-1)*overlap)
    fig, axes = plt.subplots(2, (n_panels+1)//2, figsize=(16,8))
    axes = axes.flatten()
    for i in range(n_panels):
        lo = z_min + i*step*(1-overlap); hi = lo + step
        mask = (z>=lo)&(z<=hi); ax = axes[i]
        ax.scatter(df.loc[mask,x_col], df.loc[mask,y_col], s=15, alpha=0.6)
        if mask.sum()>2:
            c = np.polyfit(df.loc[mask,x_col], df.loc[mask,y_col], 1)
            xs = np.linspace(df[x_col].min(), df[x_col].max(), 50)
            ax.plot(xs, np.polyval(c, xs), 'r-', lw=2)
            ax.set_title(f'{z_col}:[{lo:.1f},{hi:.1f}] slope={c[0]:.2f}', fontsize=9)
    for j in range(n_panels, len(axes)): axes[j].set_visible(False)
    plt.suptitle(f'Coplot: {x_col} vs {y_col} | {z_col}'); plt.tight_layout(); plt.show()

n = 500
lstat = np.random.uniform(2,35,n)
rm = 4 + 0.1*(35-lstat) + np.random.normal(0,0.5,n)
medv = 5 + 3*rm - 0.5*lstat - 0.2*rm*(lstat>20) + np.random.normal(0,2,n)
medv = np.clip(medv, 5, 50)
df_h = pd.DataFrame({'RM':rm, 'MEDV':medv, 'LSTAT':lstat})
print(f"전체 RM-MEDV 상관: {df_h['RM'].corr(df_h['MEDV']):.3f}")
coplot(df_h, 'RM', 'MEDV', 'LSTAT', n_panels=6, overlap=0.3)


**결과 해석**: Coplot으로 LSTAT 조건부에서 RM→MEDV 기울기 변화를 확인할 수 있다. 전체 상관만으로는 비선형적 조건부 관계를 포착할 수 없다.

---
## 문제 4 풀이

Trustworthiness/Continuity로 t-SNE vs MDS 정량 비교

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.manifold import TSNE, MDS
from sklearn.metrics import trustworthiness
from scipy.spatial.distance import pdist, squareform

np.random.seed(42)
clusters = [np.random.multivariate_normal(np.random.randn(10)*5, np.eye(10)*(0.5+i*0.3), sz)
            for i, sz in enumerate([200,150,100,80,50])]
X = np.vstack(clusters); labels = np.repeat(range(5), [200,150,100,80,50])

Y_mds = MDS(n_components=2, random_state=42, normalized_stress='auto').fit_transform(X)
Y_tsne = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(X)

def continuity_score(X_high, X_low, k=5):
    n = X_high.shape[0]
    D_h = squareform(pdist(X_high)); D_l = squareform(pdist(X_low))
    rank_h = np.argsort(np.argsort(D_h, axis=1), axis=1)
    total = 0
    for i in range(n):
        nn_low = set(np.argsort(D_l[i])[1:k+1])
        for j in nn_low:
            if rank_h[i,j] > k: total += rank_h[i,j] - k
    norm = n*k*(2*n-3*k-1)
    return 1 - 2/norm*total if norm>0 else 1.0

ks = [5,10,15,20,30,50]
t_mds = [trustworthiness(X, Y_mds, n_neighbors=k) for k in ks]
t_tsne = [trustworthiness(X, Y_tsne, n_neighbors=k) for k in ks]
c_mds = [continuity_score(X, Y_mds, k) for k in ks]
c_tsne = [continuity_score(X, Y_tsne, k) for k in ks]

D_o = squareform(pdist(X)); D_m = squareform(pdist(Y_mds)); D_t = squareform(pdist(Y_tsne))
tri = np.triu_indices_from(D_o, k=1)
r_mds = np.corrcoef(D_o[tri], D_m[tri])[0,1]
r_tsne = np.corrcoef(D_o[tri], D_t[tri])[0,1]

fig, axes = plt.subplots(2,2,figsize=(14,12))
axes[0,0].plot(ks,t_mds,'bo-',label='MDS'); axes[0,0].plot(ks,t_tsne,'rs-',label='t-SNE')
axes[0,0].set_title('Trustworthiness'); axes[0,0].legend(); axes[0,0].set_xlabel('k')
axes[0,1].plot(ks,c_mds,'bo-',label='MDS'); axes[0,1].plot(ks,c_tsne,'rs-',label='t-SNE')
axes[0,1].set_title('Continuity'); axes[0,1].legend(); axes[0,1].set_xlabel('k')
si = np.random.choice(len(tri[0]),5000,replace=False)
axes[1,0].scatter(D_o[tri][si],D_m[tri][si],s=1,alpha=0.3)
axes[1,0].set_title(f'Shepard: MDS (r={r_mds:.3f})')
axes[1,1].scatter(D_o[tri][si],D_t[tri][si],s=1,alpha=0.3)
axes[1,1].set_title(f'Shepard: t-SNE (r={r_tsne:.3f})')
plt.tight_layout(); plt.show()


**결과 해석**: MDS는 전역 거리 보존(높은 Shepard r)에 우수하고, t-SNE는 지역 구조(작은 k에서 높은 Trustworthiness)에 우수하다. 목적에 맞는 방법 선택이 중요하다.

---
## 확장 토론

### 시각화 선택 가이드

| 목적 | 방법 |
|------|------|
| 변수별 패턴 | 평행좌표 |
| 형태 비교 | 앤드류스 곡선 |
| 전역 구조 | MDS |
| 지역 클러스터 | t-SNE, UMAP |
| 비선형 다양체 | Isomap |
| 조건부 관계 | Coplot |